# Tree based Models

Created a new Notebook only for Tree Based models to do Feature Engineering for Supplier and Product Identifier individual for Tree Based Methods: Trees dont care about the relationship of numbers, so it is possible to just give each supplier a random number (just the way it is now, but with smaller numbers to make computation faster), Neural Networks would think that there are relationships in Suppliers with similar numbers, therefore need other feature engineering

In [48]:
# ============================================================
# Setup: Install and import all required packages
# ============================================================

# Run this once if scikit-learn is not yet installed
# !pip install scikit-learn --break-system-packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    classification_report,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [49]:
# ============================================================
# Load ML-ready datasets
# ============================================================
import pandas as pd
import numpy as np

df_company_a = pd.read_csv('../data/formated/company_a_ML.csv')
df_company_b = pd.read_csv('../data/formated/company_b_ML.csv')

print("Company A shape:", df_company_a.shape)
print("Company B shape:", df_company_b.shape)

# ============================================================
# Label Encoding for Supplier and Unique Product Identifier
# Suitable for tree-based models (Decision Tree, Random Forest, etc.)
# Trees only use these values for splits, so arbitrary numeric labels
# do not introduce a false ordering problem like they would for NNs.
# ============================================================

def label_encode_column(df, column):
    codes = {value: i for i, value in enumerate(df[column].unique())}
    df[column] = df[column].map(codes)
    return df, codes

df_company_a, supplier_codes_a = label_encode_column(df_company_a, 'Supplier')
df_company_a, product_codes_a = label_encode_column(df_company_a, 'Unique Product Identifier')

df_company_b, supplier_codes_b = label_encode_column(df_company_b, 'Supplier')
df_company_b, product_codes_b = label_encode_column(df_company_b, 'Unique Product Identifier')

df_company_a.head()

Company A shape: (4459, 20)
Company B shape: (14896, 20)


,Unique Order Identifier,Order Complexity,Supplier,Unique Product Identifier,Planned Year,Planned Month,Planned Quarter,Planned Day,Planned Weekday,Arrival Year,Arrival Month,Arrival Quarter,Arrival Day,Arrival Weekday,Delivery Delay,Delivery Status,Ordered Quantity,Delivered Quantity,Quantity Difference,Quantity Status
0,74682-11,17,0,0,2023,1,1,5,3,2022,12,4,15,3,-21,too early,3.0,3.0,0.0,exact
1,74682-14,17,0,1,2023,1,1,5,3,2022,12,4,15,3,-21,too early,2.0,2.0,0.0,exact
2,74666-2,7,1,2,2023,1,1,4,2,2023,1,1,4,2,0,on time,1.0,1.0,0.0,exact
3,74666-4,7,1,2,2023,1,1,4,2,2023,1,1,4,2,0,on time,1.0,1.0,0.0,exact
4,74806-1,1,2,3,2023,1,1,30,0,2023,1,1,4,2,-26,too early,4.0,4.0,0.0,exact


## Quality Metrics

For each prediction, we check two simple things:

**1. Precision** – when the model predicts a class (e.g. "on time"), how often
is it actually correct?

**2. Overfitting check** – we compare how well the model does on data it was
trained on vs. data it has never seen. If it does much better on the training
data, it just memorized examples instead of learning real patterns – and it
won't work well on new orders in practice.

Example:
Train Recall: 0.98 | Test Recall: 0.64 | Gap: 0.34 → High overfitting risk

A small gap (< 0.05) is fine. A large gap (> 0.15) means the model is
overfitting and needs to be simplified (e.g. lower max_depth).

In [50]:
# ============================================================
# Evaluation function - overfitting check based on macro-averaged score
# so it doesn't get hidden by an imbalanced majority class
# ============================================================
from sklearn.metrics import precision_score, recall_score

def evaluate_model(model, X_train, y_train, X_test, y_test, target_cols, model_name):
    print(f"=== {model_name} ===\n")

    for col in target_cols:
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        y_pred_train = pd.DataFrame(y_pred_train, columns=target_cols, index=y_train.index)[col]
        y_pred_test = pd.DataFrame(y_pred_test, columns=target_cols, index=y_test.index)[col]

        print(f"--- {col} ---")

        classes = sorted(y_test[col].unique())
        precisions = precision_score(y_test[col], y_pred_test, labels=classes, average=None)
        for cls, prec in zip(classes, precisions):
            print(f"Precision ({cls}): {prec:.2f}")

        # Macro recall treats every class equally, regardless of how common it is.
        # This catches cases where a model looks fine on paper (high accuracy)
        # but is actually memorizing rare classes instead of generalizing.
        train_recall = recall_score(y_train[col], y_pred_train, average='macro')
        test_recall = recall_score(y_test[col], y_pred_test, average='macro')
        gap = train_recall - test_recall

        print(f"\nTrain Recall (macro): {train_recall:.2f} | Test Recall (macro): {test_recall:.2f}")

        if gap < 0.05:
            verdict = "Low overfitting risk"
        elif gap < 0.15:
            verdict = "Moderate overfitting risk"
        else:
            verdict = "High overfitting risk"
        print(f"Overfitting check: {verdict} (gap: {gap:.2f})\n")

## Train/Test Split and Feature Selection

Target variables (y): Delivery Status, Quantity Status

Features (X): only information that is known **before** the delivery arrives.
Columns like Delivery Delay, Quantity Difference, Delivered Quantity, and all
Arrival Date features are excluded, since they are only known after the delivery
has already happened and would leak the answer directly into the model.

In [51]:
# ============================================================
# Train/Test Split - Company A and Company B
# ============================================================
from sklearn.model_selection import train_test_split

feature_cols = [
    'Order Complexity', 'Supplier', 'Unique Product Identifier',
    'Planned Year', 'Planned Month', 'Planned Quarter', 'Planned Day', 'Planned Weekday',
    'Ordered Quantity'
]
target_cols = ['Delivery Status', 'Quantity Status']

# Company A
X_a = df_company_a[feature_cols]
y_a = df_company_a[target_cols]
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a, test_size=0.2, random_state=42
)

# Company B
X_b = df_company_b[feature_cols]
y_b = df_company_b[target_cols]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42
)

print("Company A - Train:", X_train_a.shape, "Test:", X_test_a.shape)
print("Company B - Train:", X_train_b.shape, "Test:", X_test_b.shape)

Company A - Train: (3567, 9) Test: (892, 9)
Company B - Train: (11916, 9) Test: (2980, 9)


## Grid Search

In [52]:
# ============================================================
# Grid Search setup - shared scorer for multi-output models
# Uses macro recall (averaged across both target columns) so the
# grid search optimizes for the same thing evaluate_model checks
# ============================================================
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

def macro_recall_multioutput(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = [
        recall_score(y_true[:, i], y_pred[:, i], average='macro')
        for i in range(y_true.shape[1])
    ]
    return np.mean(scores)

multioutput_scorer = make_scorer(macro_recall_multioutput)

## Decision Tree

A Decision Tree is a simple supervised learning model that makes predictions
by splitting the data into branches based on feature values. Each split is a
decision rule, and the final leaf gives the predicted class.

Main parameters:
- **max_depth**: limits how deep the tree grows (controls overfitting)
- **min_samples_split / min_samples_leaf**: minimum samples required to split/form a leaf
- **criterion**: function used to measure split quality (Gini or entropy)

Possible issues: Decision Trees can easily overfit and are sensitive to small
changes in the data.

In [37]:
# ============================================================
# Decision Tree - Company A
# ============================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier

dt_model_a = MultiOutputClassifier(
    DecisionTreeClassifier(random_state=42, max_depth=20)
)
dt_model_a.fit(X_train_a, y_train_a)

y_pred_dt_a = evaluate_model(dt_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Decision Tree - Company A")

=== Decision Tree - Company A ===

--- Delivery Status ---
Precision (on time): 0.85
Precision (too early): 0.84
Precision (too late): 0.93

Train Recall (macro): 0.99 | Test Recall (macro): 0.88
Overfitting check: Moderate overfitting risk (gap: 0.11)

--- Quantity Status ---
Precision (exact): 0.99
Precision (over delivered): 1.00
Precision (under delivered): 0.50

Train Recall (macro): 0.93 | Test Recall (macro): 0.64
Overfitting check: High overfitting risk (gap: 0.29)



In [53]:
# ============================================================
# Decision Tree - Company A (Grid Search)
# ============================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.multioutput import MultiOutputClassifier

param_grid_dt = {
    'estimator__max_depth': [15, 20, 25, None],
    'estimator__min_samples_split': [2, 5, 10],
    'estimator__min_samples_leaf': [1, 2, 4],
    'estimator__criterion': ['gini', 'entropy']
}

dt_grid_a = GridSearchCV(
    MultiOutputClassifier(DecisionTreeClassifier(random_state=42)),
    param_grid=param_grid_dt,
    scoring=multioutput_scorer,
    cv=5,
    n_jobs=-1
)
dt_grid_a.fit(X_train_a, y_train_a)

print("Best hyperparameters (Decision Tree - Company A):", dt_grid_a.best_params_)
dt_model_a = dt_grid_a.best_estimator_

evaluate_model(dt_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Decision Tree - Company A")

Best hyperparameters (Decision Tree - Company A): {'estimator__criterion': 'entropy', 'estimator__max_depth': 25, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2}
=== Decision Tree - Company A ===

--- Delivery Status ---
Precision (on time): 0.85
Precision (too early): 0.84
Precision (too late): 0.92

Train Recall (macro): 0.99 | Test Recall (macro): 0.88
Overfitting check: Moderate overfitting risk (gap: 0.11)

--- Quantity Status ---
Precision (exact): 0.99
Precision (over delivered): 0.94
Precision (under delivered): 1.00

Train Recall (macro): 0.93 | Test Recall (macro): 0.64
Overfitting check: High overfitting risk (gap: 0.29)



In [41]:
# ============================================================
# Decision Tree - Company B
# ============================================================
dt_model_b = MultiOutputClassifier(
    DecisionTreeClassifier(random_state=42, max_depth=15)
)
dt_model_b.fit(X_train_b, y_train_b)

evaluate_model(dt_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "Decision Tree - Company B")

=== Decision Tree - Company B ===

--- Delivery Status ---
Precision (on time): 0.55
Precision (too early): 0.67
Precision (too late): 0.73

Train Recall (macro): 0.84 | Test Recall (macro): 0.62
Overfitting check: High overfitting risk (gap: 0.22)

--- Quantity Status ---
Precision (exact): 0.96
Precision (over delivered): 0.78
Precision (under delivered): 0.30

Train Recall (macro): 0.87 | Test Recall (macro): 0.63
Overfitting check: High overfitting risk (gap: 0.24)



## Random Forest

A Random Forest is an ensemble model that combines many Decision Trees. Each
tree is trained on a random subset of the data, and the final prediction is
made by majority vote across all trees. This usually makes it more accurate
and more stable than a single Decision Tree.

Main parameters:
- **n_estimators**: number of trees in the forest
- **max_depth**: maximum depth of each tree
- **class_weight='balanced'**: gives more weight to rare classes, which is
  important here since both targets are imbalanced

In [46]:
# ============================================================
# Random Forest - Company A
# ============================================================
from sklearn.ensemble import RandomForestClassifier

rf_model_a = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, max_depth=25, random_state=42,
                            class_weight='balanced', n_jobs=-1)
)
rf_model_a.fit(X_train_a, y_train_a)

evaluate_model(rf_model_a, X_train_a, y_train_a, X_test_a, y_test_a, target_cols, "Random Forest - Company A")

=== Random Forest - Company A ===

--- Delivery Status ---
Precision (on time): 0.92
Precision (too early): 0.85
Precision (too late): 0.95

Train Recall (macro): 0.99 | Test Recall (macro): 0.92
Overfitting check: Moderate overfitting risk (gap: 0.07)

--- Quantity Status ---
Precision (exact): 0.99
Precision (over delivered): 1.00
Precision (under delivered): 0.25

Train Recall (macro): 1.00 | Test Recall (macro): 0.67
Overfitting check: High overfitting risk (gap: 0.32)



In [47]:
# ============================================================
# Random Forest - Company B
# ============================================================
rf_model_b = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42,
                            class_weight='balanced', n_jobs=-1)
)
rf_model_b.fit(X_train_b, y_train_b)

evaluate_model(rf_model_b, X_train_b, y_train_b, X_test_b, y_test_b, target_cols, "Random Forest - Company B")

=== Random Forest - Company B ===

--- Delivery Status ---
Precision (on time): 0.60
Precision (too early): 0.72
Precision (too late): 0.80

Train Recall (macro): 0.96 | Test Recall (macro): 0.71
Overfitting check: High overfitting risk (gap: 0.25)

--- Quantity Status ---
Precision (exact): 0.97
Precision (over delivered): 0.67
Precision (under delivered): 0.38

Train Recall (macro): 0.99 | Test Recall (macro): 0.74
Overfitting check: High overfitting risk (gap: 0.25)



# Work in Progress

## Feature Importance

Random Forests allow us to see which features had the most influence on the
predictions. This directly answers one of the project's research questions:
"What are the key features that influence delivery delay and quality?"

In [ ]:
# ============================================================
# Feature Importance - Random Forest
# ============================================================
def plot_feature_importance(model, feature_cols, target_cols, company_name):
    for i, col in enumerate(target_cols):
        importances = model.estimators_[i].feature_importances_
        importance_df = pd.DataFrame({
            'Feature': feature_cols,
            'Importance': importances
        }).sort_values('Importance', ascending=False)

        plt.figure(figsize=(8, 4))
        plt.barh(importance_df['Feature'], importance_df['Importance'])
        plt.gca().invert_yaxis()
        plt.title(f"{company_name} - Feature Importance for {col}")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.show()

plot_feature_importance(rf_model_a, feature_cols, target_cols, "Company A")
plot_feature_importance(rf_model_b, feature_cols, target_cols, "Company B")